## 03 — Patient split + modern model training

1. **`make split`** — writes `data/processed/splits/split_v1.json` (needs enough labeled subjects in `results/dataset_index.csv`).
2. **`make modern_train`** — trains under `results/modern/<OPTION>/`.

Override options on the command line, e.g. `OPTION=optionB FUSION=attn_pool EPOCHS=5`.

In [1]:
import os
import subprocess
from pathlib import Path


def _scd_octa_repo_root() -> Path:
    """Jupyter cwd is often notebooks/ — `Path('..')` would point at Hospital/, not this repo."""
    p = Path.cwd().resolve()
    if (p / "Makefile").is_file() and (p / "src" / "scd_octa").is_dir():
        return p
    up = p.parent
    if (up / "Makefile").is_file() and (up / "src" / "scd_octa").is_dir():
        return up
    nested = p / "scd-octa-screening"
    if nested.is_dir() and (nested / "Makefile").is_file():
        return nested.resolve()
    raise RuntimeError(
        "Cannot find scd-octa-screening (Makefile + src/scd_octa). "
        "Run this notebook with cwd inside scd-octa-screening or its notebooks/ folder."
    )


ROOT = _scd_octa_repo_root()
os.chdir(ROOT)
print("cwd:", ROOT)

subprocess.run(["make", "split"], check=True)
subprocess.run(
    ["make", "modern_train", "OPTION=optionA", "FUSION=concat_mlp", "EPOCHS=10", "BATCH=8"],
    check=True,
)

cwd: /Users/tripa/Desktop/Projects/Hospital/scd-octa-screening
PYTHONPATH=./src /Users/tripa/Desktop/Projects/Hospital/scd-octa-screening/.venv/bin/python -m scd_octa.splits --labels-csv "./results/dataset_index.csv" --out "./data/processed/splits/split_v1.json" --seed 42 --test-size 0.2 --val-size 0.2
Wrote: data/processed/splits/split_v1.json
n_subjects: train=12 val=3 test=4
PYTHONPATH=./src /Users/tripa/Desktop/Projects/Hospital/scd-octa-screening/.venv/bin/python -m scd_octa.modern_model.train \
		--data-root "./data/raw/scd-data" \
		--labels-csv "./results/dataset_index.csv" \
		--split-json "./data/processed/splits/split_v1.json" \
		--out-dir "./results/modern/optionA" \
		--option optionA \
		--fusion concat_mlp \
		 \
		--epochs 10 \
		--batch-size 8 \
		--target-sensitivity 0.95


Wrote: results/modern/optionA/train_report.json


CompletedProcess(args=['make', 'modern_train', 'OPTION=optionA', 'FUSION=concat_mlp', 'EPOCHS=10', 'BATCH=8'], returncode=0)